In [1]:
%load_ext autoreload
%autoreload 2

from math import pi
import itertools

import numpy as np
import tqdm.contrib.itertools
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.spectrum.autodiff import FunctionSum, SampledFunction
from fluxoniumcr.spectrum.square_spectrum import SquareSpectrum

In [2]:
parent_path = DATA_DIR/"charge_operator/EJ=4.00,EC=1.20,EL=0.40"
dataset_path = parent_path/f"esd_rasterized.hdf5"

esd_dataset = xr.load_dataset(parent_path/"esd.hdf5")

In [3]:
matrix_element_spectrum = xr.DataArray(
    None,
    coords=[
        esd_dataset.drive_frequency,
        esd_dataset.deltap,
        esd_dataset.harmonic,
        esd_dataset.bra,
        esd_dataset.ket,
    ],
)

for drive_freq, deltap, harmonic, bra, ket in tqdm.contrib.itertools.product(
        matrix_element_spectrum.drive_frequency.data,
        matrix_element_spectrum.deltap.data,
        matrix_element_spectrum.harmonic.data,
        matrix_element_spectrum.bra.data,
        matrix_element_spectrum.ket.data,
):
    key = dict(
        drive_frequency=drive_freq,
        deltap=deltap,
        harmonic=harmonic,
        bra=bra,
        ket=ket,
    )
    ds = esd_dataset.loc[key]
    pole = ds.pole.item()
    bare_pole = ds.bare_pole.item()
    numerator = SampledFunction(
        pole + ds.fourier_frequency.data,
        ds.numerator.data.ravel(),
    )
    spec = SquareSpectrum(
        numerator,
        pole=pole,
        bare_pole=bare_pole,
    )
    matrix_element_spectrum.loc[key] = spec

  0%|          | 0/216288 [00:00<?, ?it/s]

In [4]:
dataset = xr.Dataset()

dataset = xr.Dataset(
    attrs=esd_dataset.attrs
)

dataset['amplitude'] = esd_dataset.amplitude

fourier_frequency_data = np.linspace(-8, 1.6, 8 * esd_dataset.drive_frequency.size) * 2*pi
fourier_frequency_coord = xr.DataArray(
    fourier_frequency_data,
    dims=['fourier_frequency'],
    attrs=dict(
        long_name="Fourier frequency",
        units="Grad/s",
    )
)
control_coord = xr.DataArray(
    np.array([0, 1], dtype=np.int8),
    dims=['control'],
    attrs=dict(
        long_name="Control qubit state",
    )
)

dataset['heatmap'] = xr.DataArray(
    np.float32('nan'),
    coords=[
        esd_dataset.drive_frequency,
        esd_dataset.deltap,
        fourier_frequency_coord,
        control_coord,
    ],
    attrs=dict(
        long_name="Rasterized spectrum",
        units="(Grad/s)-2",
    )
)

In [5]:
for drive_freq, deltap in tqdm.contrib.itertools.product(
        dataset.drive_frequency.data,
        dataset.deltap.data,
):
    for control in dataset.control.data:
        charge_op_esd = FunctionSum([
            matrix_element_spectrum.sel(
                drive_frequency=drive_freq,
                deltap=deltap,
                harmonic=harmonic,
                bra=bra,
                ket=control,
            ).item()
            for harmonic, bra in itertools.product(
                matrix_element_spectrum.harmonic,
                matrix_element_spectrum.bra,
            )
        ])
        eps = 2*pi * 1e-3  # Hide really narrow peaks.
        dataset.heatmap.loc[dict(
            drive_frequency=drive_freq,
            deltap=deltap,
            control=control,
        )][:] = charge_op_esd(fourier_frequency_data + eps*1j)

  0%|          | 0/2253 [00:00<?, ?it/s]

In [53]:
dataset.to_netcdf(dataset_path)